As GPT-2 is a encoder-only model we used: AutoModelForCausalLM. These models are called **decoder-only / causal LM**.

***AutoModel*** → selects the appropriate model implementation/architecture.

***from_pretrained()*** → loads the model configuration and pretrained weights

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

tokenizer() converts the input text into tokens and then maps those tokens to their corresponding token IDs from GPT-2’s vocabulary.

In [24]:
text = "I am Omid Gervei"

inputs = tokenizer(text, return_tensors="pt")

print(inputs["input_ids"])

tensor([[   40,   716,   440, 13602,   402,  3760,    72]])


In [25]:
print(inputs)

{'input_ids': tensor([[   40,   716,   440, 13602,   402,  3760,    72]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}


In [26]:
for token_id in inputs["input_ids"][0]:
    print(token_id.item(), "->", repr(tokenizer.decode(token_id)))

40 -> 'I'
716 -> ' am'
440 -> ' O'
13602 -> 'mid'
402 -> ' G'
3760 -> 'erve'
72 -> 'i'


In [27]:
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

print(logits.shape)

torch.Size([1, 7, 50257])


In [28]:
next_token_logits = logits[0, -1, :]

GPT-2 produces a set of logits for every token position in the input sequence.

However, since we want to predict the next token after "stormy", we only need the logits from the ***last*** position.

Here, : selects all items in the batch, -1 selects the last token position, and the final : selects all logits across the vocabulary.

So this line extracts the logits that are relevant for predicting the next token.

In [29]:
print(next_token_logits.shape)

torch.Size([50257])


In [30]:
probabilities = F.softmax(next_token_logits, dim=-1)

print("Sum of logits:",
      next_token_logits.sum().item())

print("Sum of probabilities:",
      probabilities.sum().item())

Sum of logits: -4436753.0
Sum of probabilities: 1.0000051259994507


In [31]:
next_token_id = torch.argmax(probabilities)

next_token = tokenizer.decode(next_token_id)
next_token_probability = probabilities[next_token_id]

print("Predicted token:", repr(next_token))
print("Probability:", next_token_probability.item())

Predicted token: ','
Probability: 0.2901116609573364


In [32]:
for T in [0.5, 1.0, 1.5]:
    probs_T = F.softmax(next_token_logits / T, dim=-1)
    top_probs, top_ids = torch.topk(probs_T, 10)

    print(f"\nTemperature = {T}")

    for prob, token_id in zip(top_probs, top_ids):
        print(
            f"{prob.item()*100:5.1f}%  "
            f"{repr(tokenizer.decode(token_id))}"
        )


Temperature = 0.5
 82.1%  ','
 10.9%  '.'
  2.4%  ' and'
  0.8%  ' of'
  0.8%  ',"'
  0.7%  ' who'
  0.6%  '."'
  0.6%  "'s"
  0.3%  ' ('
  0.1%  '-'

Temperature = 1.0
 29.0%  ','
 10.6%  '.'
  5.0%  ' and'
  2.9%  ' of'
  2.9%  ',"'
  2.7%  ' who'
  2.5%  '."'
  2.4%  "'s"
  1.6%  ' ('
  0.8%  '-'

Temperature = 1.5
  4.6%  ','
  2.4%  '.'
  1.4%  ' and'
  1.0%  ' of'
  1.0%  ',"'
  0.9%  ' who'
  0.9%  '."'
  0.9%  "'s"
  0.7%  ' ('
  0.4%  '-'


In [33]:
sampled_id = torch.multinomial(
    probabilities,
    num_samples=1
)

print(tokenizer.decode(sampled_id))

,


In [42]:
text = "I am Omid Gevei."

for _ in range(25):

    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[0, -1, :]

    probs = F.softmax(logits / 1.0, dim=-1)

    next_id = torch.multinomial(probs, 1)

    next_token = tokenizer.decode(next_id)

    text += next_token

    print(text)

I am Omid Gevei. I
I am Omid Gevei. I have
I am Omid Gevei. I have served
I am Omid Gevei. I have served four
I am Omid Gevei. I have served four civil
I am Omid Gevei. I have served four civil servants
I am Omid Gevei. I have served four civil servants in
I am Omid Gevei. I have served four civil servants in both
I am Omid Gevei. I have served four civil servants in both political
I am Omid Gevei. I have served four civil servants in both political parties
I am Omid Gevei. I have served four civil servants in both political parties so
I am Omid Gevei. I have served four civil servants in both political parties so I
I am Omid Gevei. I have served four civil servants in both political parties so I am
I am Omid Gevei. I have served four civil servants in both political parties so I am not
I am Omid Gevei. I have served four civil servants in both political parties so I am not very
I am Omid Gevei. I have served four civil servants in both political parties so I am not very é
I am Omid Ge